# Library & Data import

In [1]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

In [ ]:
df = pd.read_csv('Data\\311_2025_Jan_Dec.csv', low_memory=False)

In [4]:
pd.set_option('display.max_rows', 200)  # In case rows are being cut off too

# Remodeling our data

Our research focuses on segregating different areas according to their complaint type , the agency responsible for resolving the complaint , the reslve time itself , and the location. We chose to keep the columns below and remove the other columns since they dont provide significat information for the model in clustering the areas. We also removed Latitude and Longitude since Location column already has all the 

In [19]:
Rem_df = df[[
    "City",
    "Incident Zip",
    "Agency",
    "Problem (formerly Complaint Type)",
    "Location Type",
    "Closed Date",
    "Created Date",
    "Status",
    "Location"
]].copy()


In [20]:
Rem_df.head()

,City,Incident Zip,Agency,Problem (formerly Complaint Type),Location Type,Closed Date,Created Date,Status,Location
0,WOODSIDE,11377,NYPD,Blocked Driveway,Street/Sidewalk,12/31/2025 02:44:50 AM,12/31/2025 12:30:26 AM,Closed,POINT (-73.907196579197 40.743018746122)
1,FAR ROCKAWAY,11694,HPD,HEAT/HOT WATER,RESIDENTIAL BUILDING,01/02/2026 05:18:55 PM,12/31/2025 12:30:19 AM,Closed,POINT (-73.839021963252 40.577671995863)
2,NEW YORK,10033,HPD,HEAT/HOT WATER,RESIDENTIAL BUILDING,01/02/2026 10:07:10 AM,12/31/2025 12:30:06 AM,Closed,POINT (-73.937239002314 40.851013911318)
3,BROOKLYN,11207,HPD,HEAT/HOT WATER,RESIDENTIAL BUILDING,01/02/2026 01:49:56 PM,12/31/2025 12:30:05 AM,Closed,POINT (-73.899924654626 40.663886523056)
4,BROOKLYN,11234,DSNY,Derelict Vehicles,Street,12/31/2025 07:32:00 AM,12/31/2025 12:30:00 AM,Closed,POINT (-73.912414463066 40.625294042041)


The date in the data is of string type. Let's convert it to DateTime.

In [22]:
# Convert to datetime
Rem_df['Closed Date'] = pd.to_datetime(Rem_df['Closed Date'])
Rem_df['Created Date'] = pd.to_datetime(Rem_df['Created Date'])

C:\Users\Idan\AppData\Local\Temp\ipykernel_35832\3868502946.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  Rem_df['Created Date'] = pd.to_datetime(Rem_df['Created Date'])


In [ ]:
# Creating single column representing time difference
Rem_df["Resolution Time"] = Rem_df["Closed Date"] - Rem_df["Created Date"]

In [ ]:
# Removing the two Date columns
Rem_df = Rem_df.drop(columns=["Closed Date", "Created Date"])

In [29]:
Rem_df.head()

,City,Incident Zip,Agency,Problem (formerly Complaint Type),Location Type,Status,Location,Resolution Time
0,WOODSIDE,11377,NYPD,Blocked Driveway,Street/Sidewalk,Closed,POINT (-73.907196579197 40.743018746122),0 days 02:14:24
1,FAR ROCKAWAY,11694,HPD,HEAT/HOT WATER,RESIDENTIAL BUILDING,Closed,POINT (-73.839021963252 40.577671995863),2 days 16:48:36
2,NEW YORK,10033,HPD,HEAT/HOT WATER,RESIDENTIAL BUILDING,Closed,POINT (-73.937239002314 40.851013911318),2 days 09:37:04
3,BROOKLYN,11207,HPD,HEAT/HOT WATER,RESIDENTIAL BUILDING,Closed,POINT (-73.899924654626 40.663886523056),2 days 13:19:51
4,BROOKLYN,11234,DSNY,Derelict Vehicles,Street,Closed,POINT (-73.912414463066 40.625294042041),0 days 07:02:00


In [34]:
# We used an LLM to make a dictionary that groups the same complaint types into one.
broad_categories = {
    'Noise': [
        'Noise - Commercial', 'Noise - Residential', 'Noise', 'Noise - Street/Sidewalk', 
        'Noise - Vehicle', 'Noise - Park', 'Noise - Helicopter', 'Noise - House of Worship'
    ],
    
    'Housing & Building Maintenance': [
        'HEAT/HOT WATER', 'PLUMBING', 'FLOORING/STAIRS', 'UNSANITARY CONDITION', 'GENERAL', 
        'ELECTRIC', 'DOOR/WINDOW', 'APPLIANCE', 'Boilers', 'PAINT/PLASTER', 'ELEVATOR', 'Elevator', 
        'WATER LEAK', 'OUTSIDE BUILDING', 'Non-Residential Heat', 'Window Guard', 'Building Condition', 
        'Indoor Sewage', 'Mold'
    ],
    
    'Vehicles & Parking': [
        'Blocked Driveway', 'Derelict Vehicles', 'Illegal Parking', 'Abandoned Vehicle', 'Traffic', 
        'Abandoned Bike', 'Broken Parking Meter', 'Municipal Parking Facility', 'Bike Rack'
    ],
    
    'Street & Infrastructure': [
        'Sidewalk Condition', 'Street Light Condition', 'Street Condition', 'Root/Sewer/Sidewalk Condition', 
        'Traffic Signal Condition', 'Highway Condition', 'Bridge Condition', 'Curb Condition', 
        'Street Sign - Damaged', 'Street Sign - Missing', 'Street Sign - Dangling', 'Highway Sign - Damaged', 
        'Highway Sign - Missing', 'Highway Sign - Dangling', 'DEP Street Condition', 'DEP Sidewalk Condition', 
        'DEP Highway Condition', 'Tunnel Condition', 'Snow or Ice'
    ],
    
    'Sanitation & Trash': [
        'Dumpster Complaint', 'Dirty Condition', 'Sanitation Worker or Vehicle Complaint', 'Sewer', 
        'Illegal Dumping', 'Commercial Disposal Complaint', 'Residential Disposal Complaint', 
        'Litter Basket Complaint', 'Street Sweeping Complaint', 'Litter Basket Request', 'Graffiti',
        'Recycling Basket Complaint', 'Missed Collection', 'Institution Disposal Complaint', 
        'Transfer Station Complaint', 'DSNY Internal', 'Industrial Waste'
    ],
    
    'Animals & Pets': [
        'Dead Animal', 'Animal-Abuse', 'Unleashed Dog', 'Animal in a Park', 'Harboring Bees/Wasps', 
        'Unsanitary Animal Facility', 'Animal Facility - No Permit', 'Illegal Animal Sold', 'Pet Sale', 
        'Unsanitary Animal Pvt Property', 'Illegal Animal Kept as Pet', 'Unsanitary Pigeon Condition', 'Pet Shop'
    ],
    
    'Trees & Parks': [
        'Dead/Dying Tree', 'Damaged Tree', 'Illegal Tree Damage', 'Overgrown Tree/Branches', 
        'New Tree Request', 'Uprooted Stump', 'Violation of Park Rules', 'Plant', 'Poison Ivy', 
        'Special Natural Area District (SNAD)', 'Wood Pile Remaining'
    ],
    
    'Health & Environmental Safety': [
        'Indoor Air Quality', 'Food Establishment', 'Food Poisoning', 'Hazardous Materials', 'Lead', 
        'Asbestos', 'ASBESTOS', 'Air Quality', 'Drinking Water', 'Water Quality', 'Radioactive Material', 
        'Oil or Gas Spill', 'Cooling Tower', 'Standing Water', 'Mosquitoes', 'Building Drinking Water Tank', 
        'Water Conservation', 'Water System'
    ],
    
    'Public Order & Police': [
        'Non-Emergency Police Matter', 'Drug Activity', 'Urinating in Public', 'Smoking or Vaping', 
        'Illegal Fireworks', 'Panhandling', 'Encampment', 'Drinking', 'Disorderly Youth', 
        'Homeless Person Assistance', 'Illegal Posting', 'Posting Advertisement'
    ],
    
    'Construction & DOB': [
        'General Construction/Plumbing', 'Plumbing', 'Building/Use', 'Scaffold Safety', 'BEST/Site Safety', 
        'Construction Lead Dust', 'Cranes and Derricks', 'Construction Safety Enforcement', 'AHV Inspection Unit',
        'Special Operations', 'Borough Office'
    ],
    
    'Taxi & For-Hire Vehicles': [
        'Taxi Complaint', 'For Hire Vehicle Complaint', 'Taxi Report', 'For Hire Vehicle Report', 
        'Green Taxi Complaint', 'Taxi Compliment', 'Green Taxi Report', 'Taxi Licensee Complaint', 
        'FHV Licensee Complaint'
    ]
}

# This creates a dict that looks like {'Noise - Commercial': 'Noise', 'HEAT/HOT WATER': 'Housing & Building Maintenance', ...}
mapping_dict = {
    specific_type: broad_category 
    for broad_category, specific_types_list in broad_categories.items() 
    for specific_type in specific_types_list
}

# Applying the mapping to our dataframe
Rem_df['Complaint_Type'] = Rem_df['Problem (formerly Complaint Type)'].map(mapping_dict).fillna('Other/Misc')

# 4. Verify the new clean categories
print(Rem_df['Complaint_Type'].value_counts())

Complaint_Type
Vehicles & Parking                877554
Noise                             831176
Housing & Building Maintenance    786282
Street & Infrastructure           229255
Sanitation & Trash                224815
Public Order & Police             179686
Other/Misc                        177515
Health & Environmental Safety     141139
Trees & Parks                      71259
Construction & DOB                 63241
Animals & Pets                     34471
Taxi & For-Hire Vehicles           29602
Name: count, dtype: int64


In [ ]:
# Dropping previous unorganized complaint column
Rem_df = Rem_df.drop("Problem (formerly Complaint Type)", axis = 1 , inplace = False)

In [ ]:
# This is our final table 
Rem_df.head()

,City,Incident Zip,Agency,Location Type,Status,Location,Resolution Time,Complaint_Type
0,WOODSIDE,11377,NYPD,Street/Sidewalk,Closed,POINT (-73.907196579197 40.743018746122),0 days 02:14:24,Vehicles & Parking
1,FAR ROCKAWAY,11694,HPD,RESIDENTIAL BUILDING,Closed,POINT (-73.839021963252 40.577671995863),2 days 16:48:36,Housing & Building Maintenance
2,NEW YORK,10033,HPD,RESIDENTIAL BUILDING,Closed,POINT (-73.937239002314 40.851013911318),2 days 09:37:04,Housing & Building Maintenance
3,BROOKLYN,11207,HPD,RESIDENTIAL BUILDING,Closed,POINT (-73.899924654626 40.663886523056),2 days 13:19:51,Housing & Building Maintenance
4,BROOKLYN,11234,DSNY,Street,Closed,POINT (-73.912414463066 40.625294042041),0 days 07:02:00,Vehicles & Parking


In [41]:
# Let's save the table as csv
Rem_df.to_csv("Organized_data_1_0.csv", index=False)